# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: A page is worth reviewing if it's stale (hasn't been updated in 180+ days) but still gets meaningful search visibility, OR if it ranks well but has surprisingly low click-through, OR if it's declining and still has real demand. Every flagged page gets a reason code explaining exactly why it was picked, so a reviewer never has to guess.

Reason codes my rule can output:

stale_visible_page — old, unrefreshed, but still seen by real searchers
low_ctr_visible_page — ranks well but isn't earning the clicks it should
declining_with_demand — actively losing traffic while still in demand
thin_visible_page — short/thin content still pulling in visibility
general_refresh_review — flagged for review but doesn't cleanly match a specific pattern

In [ ]:
import pandas as pd

df = pd.read_csv('FlyRank-Internship/data/raw/content_refresh_anonymized.csv')

# Bucket pages into "stale" vs "fresh" based on days since last update
df['staleness_bucket'] = df['days_since_last_update'].apply(
    lambda x: 'stale (180+ days)' if x >= 180 else 'fresh (<180 days)'
)

staleness_table = df.groupby('staleness_bucket').agg(
    n=('content_id', 'count'),
    declining_rate=('trend_direction', lambda x: (x == 'down').mean())
).round(3)

print(staleness_table)

In [ ]:
# Bucket by whether a page has good position but still low clicks
df['ctr_position_bucket'] = df.apply(
    lambda row: 'good_position_low_ctr' if (0 < row['avg_position'] <= 20 and row['ctr'] < 0.5)
    else 'other',
    axis=1
)

ctr_table = df.groupby('ctr_position_bucket').agg(
    n=('content_id', 'count'),
    declining_rate=('trend_direction', lambda x: (x == 'down').mean())
).round(3)

print(ctr_table)

In [ ]:
print("Signal 1 (staleness) verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE — write based on the numbers above]")
print("Signal 2 (CTR vs position) verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE — write based on the numbers above]")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np

# --- Step 1: build 4 sub-scores, each between 0 and 1 ---

# Visibility: how much search exposure does the page get? (log-scaled so huge outliers don't dominate)
df['visibility_score'] = np.log1p(df['impressions_90d']).rank(pct=True)

# Freshness risk: how overdue is this page for an update?
df['freshness_risk_score'] = df['days_since_last_update'].rank(pct=True)

# Position opportunity: good position (low number) + high visibility = valuable to protect
df['position_opportunity_score'] = (
    (1 - df['avg_position'].clip(lower=1, upper=50).rank(pct=True))
    * df['visibility_score']
    * (df['avg_position'] > 0).astype(int)
)

# Depth gap: thin content that still gets traffic is a quick-win opportunity
df['depth_gap_score'] = (1 - df['word_count'].rank(pct=True)) * df['visibility_score']

# --- Step 2: combine into ONE final score using weights ---
df['baseline_action_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opportunity_score']
    + 0.05 * df['depth_gap_score']
).clip(0, 1)

# --- Step 3: assign reason codes ---
def get_reason_codes(row):
    reasons = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        reasons.append('stale_visible_page')
    if row['trend_direction'] == 'down' and row['impressions_90d'] >= 100:
        reasons.append('declining_with_demand')
    if pd.notna(row['word_count']) and row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('thin_visible_page')
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')
    if not reasons:
        reasons.append('general_refresh_review')
    return '|'.join(reasons)

df['reason_codes'] = df.apply(get_reason_codes, axis=1)

# --- Step 4: suggest an action based on the reason codes ---
def suggest_action(reason_codes):
    reasons = reason_codes.split('|')
    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in reasons or 'declining_with_demand' in reasons:
        return 'refresh'
    return 'monitor'

df['suggested_action'] = df['reason_codes'].apply(suggest_action)

# --- Step 5: rank everything, best (highest score) first ---
df['baseline_rank'] = df['baseline_action_score'].rank(method='first', ascending=False).astype(int)
ranked = df.sort_values('baseline_rank')

# --- Step 6: write the CSV your assignment requires ---
import os
os.makedirs('work/outputs', exist_ok=True)
output_cols = ['content_id', 'client_id', 'baseline_rank', 'baseline_action_score',
               'reason_codes', 'suggested_action', 'impressions_90d', 'avg_position',
               'ctr', 'days_since_last_update', 'word_count', 'trend_direction']
ranked[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Saved", len(ranked), "ranked rows to work/outputs/baseline_action_score.csv")
ranked[output_cols].head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = ranked[output_cols].head(20).reset_index(drop=True)
top20

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Looking at your 20 hand-reviewed rows, write which 2-3 look the weakest and why — for example, a page flagged general_refresh_review with no specific reason code is inherently weaker evidence than one with three stacked reasons.

In [ ]:
# Confirm the label itself was NEVER used to build the score
leak_check = ['baseline_action_score', 'visibility_score', 'freshness_risk_score',
              'position_opportunity_score', 'depth_gap_score', 'reason_codes']

leaked = [col for col in leak_check if 'trend' in col.lower()]
print("Leakage check — any trend/label columns used in scoring:", leaked if leaked else "NONE FOUND — clean")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.